<a href="https://colab.research.google.com/github/AqiqiySalimuzzaakiy/Alpro/blob/main/IYD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

!pip install ultralytics roboflow PyYAML -q
print("Instalasi library ultralytics, roboflow, dan PyYAML selesai.")

import os
import time
import shutil
import yaml
from pathlib import Path
from roboflow import Roboflow
from ultralytics import YOLO
from google.colab import userdata

print("Semua library dasar telah diimport.")

Sun Mar 29 13:13:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
try:
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
    print("API Key Roboflow berhasil diambil dari Colab Secrets.")
except userdata.SecretNotFoundError:
    print("Secret 'ROBOFLOW_API_KEY' tidak ditemukan. Menggunakan API key manual.")
    ROBOFLOW_API_KEY = "MASUKKAN_API_KEY_ROBOFLOW_KAMU_DI_SINI"
    if ROBOFLOW_API_KEY == "MASUKKAN_API_KEY_ROBOFLOW_KAMU_DI_SINI" or not ROBOFLOW_API_KEY:
        raise ValueError("API Key Roboflow belum diisi. Harap perbarui.")
    else:
        print("Menggunakan API Key manual (hati-hati jika notebook dibagikan).")
except Exception as e:
    raise ValueError(f"Terjadi kesalahan saat mengambil API Key: {e}")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
print("Objek Roboflow berhasil diinisialisasi.")

MY_WORKSPACE_ID = "aqiqiy-salimuzzaakiy"

projects_info = [
    (MY_WORKSPACE_ID, "pothole-detection-2-cqgmw", 1, "pothole_detection"),
    (MY_WORKSPACE_ID, "night-detection-gntpo", 1, "night_detection"),
    (MY_WORKSPACE_ID, "rain-detection", 1, "rain_detection"),
    (MY_WORKSPACE_ID, "accident-detection-2-ax2do", 1, "accident_detection"),
    (MY_WORKSPACE_ID, "object-detection-0iwfn", 1, "general_object_detection")
]
print("\nDefinisi dataset individual yang akan diunduh:")
for ws, proj, ver, name in projects_info:
    print(f"- Nama Lokal: {name}, Project ID: {proj}, Versi: {ver}")

BASE_DIR = Path("/content/yolo_generalist_project")
BASE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Direktori utama project: {BASE_DIR}")

API Key Roboflow berhasil diambil dari Colab Secrets.
Objek Roboflow berhasil diinisialisasi.

Definisi dataset individual yang akan diunduh:
- Nama Lokal: pothole_detection, Project ID: pothole-detection-2-cqgmw, Versi: 1
- Nama Lokal: night_detection, Project ID: night-detection-gntpo, Versi: 1
- Nama Lokal: rain_detection, Project ID: rain-detection, Versi: 1
- Nama Lokal: accident_detection, Project ID: accident-detection-2-ax2do, Versi: 1
- Nama Lokal: general_object_detection, Project ID: object-detection-0iwfn, Versi: 1
Direktori utama project: /content/yolo_generalist_project


In [ ]:
dataset_yaml_paths = {}
downloaded_dataset_roots = {}
debug_first_dataset_structure = True

print("\n--- Memulai Proses Pengunduhan Dataset Individual ---")
for workspace_id, project_id, version_number, local_name in projects_info:
    try:
        print(f"\nMengunduh dataset untuk: '{local_name}' (Project ID: {project_id}, Versi: {version_number})")
        project_rf = rf.workspace(workspace_id).project(project_id)
        version_rf = project_rf.version(version_number)

        print(f"  Memulai download dari Roboflow...")
        downloaded_dataset_info = version_rf.download("yolov8")

        dataset_root_path = Path(downloaded_dataset_info.location)
        print(f"  Selesai download. Lokasi dataset dari Roboflow: {dataset_root_path}")
        downloaded_dataset_roots[local_name] = dataset_root_path

        if debug_first_dataset_structure and dataset_root_path.is_dir():
            print(f"  [DEBUG] Isi dari direktori unduhan '{dataset_root_path}' (untuk '{local_name}'):")
            for item_path in sorted(list(dataset_root_path.rglob("*"))):
                 print(f"    {item_path.relative_to(dataset_root_path)}")
            debug_first_dataset_structure = False

        yaml_path_found = dataset_root_path / "data.yaml"
        if yaml_path_found.exists():
            dataset_yaml_paths[local_name] = yaml_path_found
            print(f"  -> BERHASIL. Path data.yaml ditemukan: {yaml_path_found}")
        else:
            dataset_yaml_paths[local_name] = None
            print(f"  -> PERINGATAN: File 'data.yaml' tidak ditemukan di {dataset_root_path}.")
    except Exception as e:
        print(f"  -> GAGAL pada proses untuk dataset '{local_name}'. Error: {e}")
        dataset_yaml_paths[local_name] = None
        downloaded_dataset_roots[local_name] = None
    time.sleep(1)

print("\n--- Ringkasan Hasil Pengunduhan Dataset Individual (Lokasi data.yaml) ---")
for name, location in dataset_yaml_paths.items():
    status = str(location) if location else 'GAGAL UNDUH / data.yaml TIDAK DITEMUKAN'
    print(f"- Nama Lokal Dataset: '{name}' -> Path data.yaml: {status}")

if not all(dataset_yaml_paths.values()):
    print("\nPERINGATAN: Tidak semua dataset berhasil diunduh atau data.yaml tidak ditemukan. Proses penggabungan mungkin tidak lengkap.")


--- Memulai Proses Pengunduhan Dataset Individual ---

Mengunduh dataset untuk: 'pothole_detection' (Project ID: pothole-detection-2-cqgmw, Versi: 1)
loading Roboflow workspace...
loading Roboflow project...
  Memulai download dari Roboflow...
  Selesai download. Lokasi dataset dari Roboflow: /content/Pothole-detection-2-1
  [DEBUG] Isi dari direktori unduhan '/content/Pothole-detection-2-1' (untuk 'pothole_detection'):
    README.dataset.txt
    README.roboflow.txt
    data.yaml
    test
    test/images
    test/images/S8_jpeg.rf.9d83b58c76822affa566e9668eaa1dd9.jpg
    test/images/S8_jpeg_jpg.rf.fec44d75f390ea813b6f92ebb7f8128f.jpg
    test/images/images164_jpg.rf.132cceaecd1a49243bc3b3f09ddd9e01.jpg
    test/images/images164_jpg.rf.add0959ae4a7bd63364e1419c221b7c5.jpg
    test/images/images173_jpg.rf.8880b19f4624174085e531662ad440d1.jpg
    test/images/images173_jpg.rf.acc2e741592cfab14c19ee1ae4ebda77.jpg
    test/images/images175_jpg.rf.483e960430fe289f105ba87ebef838b2.jpg
    tes

In [ ]:
# Cell 4 (REVISI PALING BARU): Menggabungkan Dataset Secara Manual

print("\n--- Memulai Proses Penggabungan Dataset Manual (Revisi Paling Baru) ---")

# Pastikan variabel dari sel sebelumnya ada (sama seperti sebelumnya)
if 'dataset_yaml_paths' not in globals() or not dataset_yaml_paths:
    raise ValueError("Variabel 'dataset_yaml_paths' tidak lengkap. Jalankan Cell 3.")
if 'downloaded_dataset_roots' not in globals() or not downloaded_dataset_roots:
    raise ValueError("Variabel 'downloaded_dataset_roots' tidak lengkap. Jalankan Cell 3.")
if 'BASE_DIR' not in globals():
    raise ValueError("Variabel 'BASE_DIR' tidak ada. Jalankan Cell 2.")

COMBINED_DATA_DIR = BASE_DIR / "combined_dataset"
COMBINED_IMAGES_DIR = COMBINED_DATA_DIR / "images"
COMBINED_LABELS_DIR = COMBINED_DATA_DIR / "labels"
COMBINED_TRAIN_IMG_DIR = COMBINED_IMAGES_DIR / "train"
COMBINED_VALID_IMG_DIR = COMBINED_IMAGES_DIR / "valid"
COMBINED_TRAIN_LBL_DIR = COMBINED_LABELS_DIR / "train"
COMBINED_VALID_LBL_DIR = COMBINED_LABELS_DIR / "valid"

for p in [COMBINED_TRAIN_IMG_DIR, COMBINED_VALID_IMG_DIR, COMBINED_TRAIN_LBL_DIR, COMBINED_VALID_LBL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

global_class_names_list = []
global_class_name_to_id_map = {}
current_global_class_id = 0
dataset_configs_processed = {}

print("\n1. Mengumpulkan nama kelas unik dan memproses konfigurasi dataset...")
# (Blok normalisasi kelas sama persis seperti di "Revisi Ultimate", jadi saya singkat di sini)
# --- AWAL BLOK NORMALISASI KELAS (SALIN DARI REVISI ULTIMATE) ---
for local_name, original_yaml_path in dataset_yaml_paths.items():
    if not original_yaml_path or not original_yaml_path.exists():
        print(f"  Peringatan: Path YAML untuk '{local_name}' tidak valid. Dilewati.")
        continue
    with open(original_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)
    original_class_names_from_yaml = data_config.get('names', [])
    processed_names_for_this_dataset = list(original_class_names_from_yaml)
    # ... (SISIPKAN SEMUA LOGIKA NORMALISASI KELAS DARI REVISI ULTIMATE DI SINI) ...
    # Contoh sederhana (harus dilengkapi):
    if local_name == 'pothole_detection':
        temp_names = [name for name in processed_names_for_this_dataset if name not in ['------------------------------', 'Pothole Detection Project - v8 2025-01-29 4-26pm', 'Roboflow is an end-to-end computer vision platform that helps you']]
        normalized_names = []
        p_added = False; obj_added = False
        for name in temp_names:
            if name.lower() == 'pothole':
                if not p_added: normalized_names.append('pothole'); p_added = True
            elif name.lower() == 'object': # Tangani 'object' jika ada
                 if not obj_added: normalized_names.append('object'); obj_added = True
            elif name not in normalized_names : normalized_names.append(name)
        processed_names_for_this_dataset = normalized_names

    temp_normalized_overall = []
    flags = {'car': False, 'bus': False, 'truck': False, 'bicycle': False, 'motorbike': False}
    key_map = {'cars': 'car', 'motorcycle': 'motorbike'}
    for name in processed_names_for_this_dataset:
        name_lower = name.lower()
        std_name = key_map.get(name_lower, name_lower)
        processed_this_iteration = False
        if std_name in flags:
            if not flags[std_name]: temp_normalized_overall.append(std_name); flags[std_name] = True
            processed_this_iteration = True
        elif 'bus' in std_name : # perhatikan urutan, jika 'bus' ada di flags, ini tidak akan jalan
            if not flags['bus']: temp_normalized_overall.append('bus'); flags['bus'] = True
            processed_this_iteration = True
        elif 'truck' in std_name :
            if not flags['truck']: temp_normalized_overall.append('truck'); flags['truck'] = True
            processed_this_iteration = True

        if not processed_this_iteration and std_name not in temp_normalized_overall:
             temp_normalized_overall.append(std_name)
    processed_names_for_this_dataset = temp_normalized_overall
    # --- AKHIR BLOK NORMALISASI KELAS ---
    print(f"  Dataset '{local_name}' -> kelas diproses: {processed_names_for_this_dataset}")
    dataset_configs_processed[local_name] = {
        'original_yaml_path': original_yaml_path,
        'processed_class_names': processed_names_for_this_dataset,
        'original_names_from_yaml': list(data_config.get('names', [])),
    }
    for name in processed_names_for_this_dataset:
        if name not in global_class_name_to_id_map:
            global_class_name_to_id_map[name] = current_global_class_id
            global_class_names_list.append(name)
            current_global_class_id += 1
# ... (Bagian print Total kelas unik & Daftar kelas global sama) ...
print(f"Total kelas unik global: {len(global_class_names_list)}")
print(f"Daftar kelas global: {global_class_names_list}")


print("\n2. Memproses gambar dan label untuk digabungkan...")
for local_name, config_data in dataset_configs_processed.items():
    original_id_to_name_map_local = {idx: name for idx, name in enumerate(config_data['original_names_from_yaml'])}

    print(f"  Memproses dataset: '{local_name}'...")
    dataset_root = downloaded_dataset_roots.get(local_name)
    if not dataset_root or not dataset_root.is_dir():
        print(f"    Root path untuk '{local_name}' ('{dataset_root}') tidak valid. Dilewati.")
        continue

    # ASUMSIKAN STRUKTUR FOLDER STANDAR: dataset_root / split_name / images dan dataset_root / split_name / labels
    for original_split_key in ["train", "valid"]: # Langsung coba 'train' dan 'valid'
        target_split_name = original_split_key # 'train' tetap 'train', 'valid' tetap 'valid'
        print(f"    Memproses split: '{original_split_key}' (disimpan sbg '{target_split_name}') untuk '{local_name}'")

        # Bentuk path ke direktori gambar dan label secara langsung berdasarkan asumsi struktur
        original_img_dir = (dataset_root / original_split_key / "images").resolve()
        original_lbl_dir = (dataset_root / original_split_key / "labels").resolve()

        print(f"      DEBUG: dataset_root = {dataset_root}")
        print(f"      DEBUG: Mencoba original_img_dir = {original_img_dir}")
        print(f"      DEBUG: Mencoba original_lbl_dir = {original_lbl_dir}")

        if not original_img_dir.is_dir():
            print(f"      Direktori gambar '{original_split_key}' ('{original_img_dir}') TIDAK DITEMUKAN. Dilewati.")
            continue # Lanjut ke split berikutnya atau dataset berikutnya

        target_img_dir = COMBINED_TRAIN_IMG_DIR if target_split_name == "train" else COMBINED_VALID_IMG_DIR
        target_lbl_dir = COMBINED_TRAIN_LBL_DIR if target_split_name == "train" else COMBINED_VALID_LBL_DIR

        img_files_processed_count = 0
        # ... (Sisa loop untuk menyalin gambar dan memproses label SAMA SEPERTI "REVISI ULTIMATE") ...
        # --- AWAL BAGIAN PROSES FILE (SALIN DARI REVISI ULTIMATE) ---
        for img_file_path_orig in original_img_dir.glob('*.[jp][pn]g*'):
            new_img_filename = f"{local_name}_{img_file_path_orig.name}"
            target_img_filepath = target_img_dir / new_img_filename
            shutil.copy2(img_file_path_orig, target_img_filepath)

            original_label_filename = img_file_path_orig.stem + ".txt"
            original_label_filepath = original_lbl_dir / original_label_filename
            new_label_filename = target_img_filepath.stem + ".txt"
            target_label_filepath = target_lbl_dir / new_label_filename

            if original_label_filepath.exists():
                with open(original_label_filepath, 'r') as f_orig_lbl, open(target_label_filepath, 'w') as f_new_lbl:
                    for line in f_orig_lbl:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            try:
                                original_class_id = int(parts[0])
                                original_class_name_actual = original_id_to_name_map_local.get(original_class_id)

                                if original_class_name_actual is not None:
                                    name_lower_lookup = original_class_name_actual.lower()
                                    normalized_name_for_global_lookup = original_class_name_actual
                                    if local_name == 'pothole_detection' and name_lower_lookup == 'pothole':
                                        normalized_name_for_global_lookup = 'pothole'
                                    elif name_lower_lookup == 'cars': normalized_name_for_global_lookup = 'car'
                                    elif 'bus' in name_lower_lookup: normalized_name_for_global_lookup = 'bus'
                                    elif 'truck' in name_lower_lookup: normalized_name_for_global_lookup = 'truck'
                                    elif name_lower_lookup == 'motorcycle': normalized_name_for_global_lookup = 'motorbike'
                                    elif name_lower_lookup == 'bicycle' : normalized_name_for_global_lookup = 'bicycle'

                                    global_new_class_id = global_class_name_to_id_map.get(normalized_name_for_global_lookup)

                                    if global_new_class_id is not None:
                                        new_line = f"{global_new_class_id} {' '.join(parts[1:])}\n"
                                        f_new_lbl.write(new_line)
                                    else:
                                        print(f"        WARN: Kelas ternormalisasi '{normalized_name_for_global_lookup}' (dari '{original_class_name_actual}') dataset '{local_name}' tidak ada di map global. Kelas global: {list(global_class_name_to_id_map.keys())[:5]}...")
                                else:
                                    print(f"        WARN: ID kelas asli '{original_class_id}' dari '{local_name}' tidak ada di map kelas YAML dataset itu.")
                            except ValueError:
                                print(f"        WARN: Baris tidak valid di label '{original_label_filepath}': {line.strip()}")
            img_files_processed_count += 1
        # --- AKHIR BAGIAN PROSES FILE ---
        print(f"      Selesai memproses {img_files_processed_count} gambar untuk split '{original_split_key}' dari '{local_name}'.")

print("\n3. Membuat file data.yaml untuk dataset gabungan...")
# ... (Bagian pembuatan combined_data.yaml sama seperti revisi sebelumnya) ...
# --- AWAL BAGIAN PEMBUATAN COMBINED_DATA.YAML (SAMA) ---
combined_data_yaml_content = {
    'path': str(COMBINED_DATA_DIR.resolve()),
    'train': str((COMBINED_IMAGES_DIR / "train").relative_to(COMBINED_DATA_DIR)),
    'val': str((COMBINED_IMAGES_DIR / "valid").relative_to(COMBINED_DATA_DIR)),
    'nc': len(global_class_names_list),
    'names': global_class_names_list
}
COMBINED_DATA_YAML_PATH = COMBINED_DATA_DIR / "combined_data.yaml"
with open(COMBINED_DATA_YAML_PATH, 'w') as f:
    yaml.dump(combined_data_yaml_content, f, sort_keys=False, default_flow_style=None)
print(f"File 'combined_data.yaml' berhasil dibuat di: {COMBINED_DATA_YAML_PATH}")
print("Isi combined_data.yaml:")
print(yaml.dump(combined_data_yaml_content, sort_keys=False, default_flow_style=None))
# --- AKHIR BAGIAN PEMBUATAN COMBINED_DATA.YAML ---
print("--- Proses Penggabungan Dataset Selesai ---")


--- Memulai Proses Penggabungan Dataset Manual (Revisi Paling Baru) ---

1. Mengumpulkan nama kelas unik dan memproses konfigurasi dataset...
  Dataset 'pothole_detection' -> kelas diproses: ['pothole', 'object']
  Dataset 'night_detection' -> kelas diproses: ['auto-rickshaw', 'bicycle', 'bus', 'car', 'e-rickshaw', 'motorbike', 'van']
  Dataset 'rain_detection' -> kelas diproses: ['bicycle', 'bus', 'car', 'motorbike', 'others', 'person']
  Dataset 'accident_detection' -> kelas diproses: ['major', 'moderate', 'severe']
  Dataset 'general_object_detection' -> kelas diproses: ['bus', 'truck', 'car', 'human', 'motorbike']
Total kelas unik global: 16
Daftar kelas global: ['pothole', 'object', 'auto-rickshaw', 'bicycle', 'bus', 'car', 'e-rickshaw', 'motorbike', 'van', 'others', 'person', 'major', 'moderate', 'severe', 'truck', 'human']

2. Memproses gambar dan label untuk digabungkan...
  Memproses dataset: 'pothole_detection'...
    Memproses split: 'train' (disimpan sbg 'train') untuk 'po

In [ ]:
print("Jumlah file di direktori gambar training gabungan:")
!ls -1 /content/yolo_generalist_project/combined_dataset/images/train | wc -l

print("\nJumlah file di direktori label training gabungan:")
!ls -1 /content/yolo_generalist_project/combined_dataset/labels/train | wc -l

print("\nJumlah file di direktori gambar validasi gabungan:")
!ls -1 /content/yolo_generalist_project/combined_dataset/images/valid | wc -l

print("\nJumlah file di direktori label validasi gabungan:")
!ls -1 /content/yolo_generalist_project/combined_dataset/labels/valid | wc -l

Jumlah file di direktori gambar training gabungan:
6563

Jumlah file di direktori label training gabungan:
6563

Jumlah file di direktori gambar validasi gabungan:
1563

Jumlah file di direktori label validasi gabungan:
1563


In [ ]:
# Cell 5 (BARU): Latih Satu Model Generalis

print("\n--- Memulai Pelatihan Model Generalis Tunggal ---")

# Pastikan COMBINED_DATA_YAML_PATH dari Cell 4 sudah ada
if 'COMBINED_DATA_YAML_PATH' not in globals() or not COMBINED_DATA_YAML_PATH.exists():
    raise ValueError("File 'combined_data.yaml' tidak ditemukan. Jalankan Cell 4 (Penggabungan Dataset) terlebih dahulu.")

# Parameter training (bisa disesuaikan)
EPOCHS = 50 # Atau sesuai keinginanmu
IMAGE_SIZE = 640
BATCH_SIZE = 16
YOLO_MODEL_TYPE = 'yolov8n.pt'

# Direktori untuk menyimpan hasil training model generalis
# Pastikan BASE_DIR sudah terdefinisi dari Cell 2
GENERALIST_MODEL_OUTPUT_DIR = BASE_DIR / "trained_generalist_model"
GENERALIST_TRAINING_RUN_NAME = "generalist_training_run"

try:
    model = YOLO(YOLO_MODEL_TYPE)
    print(f"Melatih model generalis dengan data: {COMBINED_DATA_YAML_PATH}")
    model.train(
        data=str(COMBINED_DATA_YAML_PATH), # Harus string
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        project=str(GENERALIST_MODEL_OUTPUT_DIR),
        name=GENERALIST_TRAINING_RUN_NAME,
        exist_ok=True
    )

    GENERALIST_BEST_PT_PATH = GENERALIST_MODEL_OUTPUT_DIR / GENERALIST_TRAINING_RUN_NAME / "weights" / "best.pt"
    if GENERALIST_BEST_PT_PATH.exists():
        print(f"Pelatihan model generalis SELESAI. Model terbaik disimpan di: {GENERALIST_BEST_PT_PATH}")
    else:
        print(f"Pelatihan model generalis mungkin selesai, TAPI 'best.pt' TIDAK DITEMUKAN di {GENERALIST_BEST_PT_PATH}")
        GENERALIST_BEST_PT_PATH = None # Set None jika tidak ditemukan

except Exception as e:
    print(f"GAGAL melakukan pelatihan model generalis. Error: {e}")
    GENERALIST_BEST_PT_PATH = None

print("--- Pelatihan Model Generalis Selesai ---")


--- Memulai Pelatihan Model Generalis Tunggal ---
Melatih model generalis dengan data: /content/yolo_generalist_project/combined_dataset/combined_data.yaml
Ultralytics 8.3.149 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_generalist_project/combined_dataset/combined_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=

train: Scanning /content/yolo_generalist_project/combined_dataset/labels/train... 6563 images, 557 backgrounds, 0 corrupt: 100%|██████████| 6563/6563 [00:02<00:00, 2267.75it/s]


train: New cache created: /content/yolo_generalist_project/combined_dataset/labels/train.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 3414, len(boxes) = 26976. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 365.8±63.4 MB/s, size: 55.2 KB)


val: Scanning /content/yolo_generalist_project/combined_dataset/labels/valid... 1563 images, 260 backgrounds, 0 corrupt: 100%|██████████| 1563/1563 [00:01<00:00, 825.23it/s]

val: New cache created: /content/yolo_generalist_project/combined_dataset/labels/valid.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 1002, len(boxes) = 6703. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      24/50      3.51G      1.276     0.9338      1.139         33        640: 100%|██████████| 411/411 [02:18<00:00,  2.97it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 49/49 [00:26<00:00,  1.83it/s]

                   all       1563       6703      0.683      0.548      0.569      0.377



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      25/50      3.51G      1.272     0.9256       1.14         56        640: 100%|██████████| 411/411 [02:27<00:00,  2.79it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):  39%|███▉      | 19/49 [00:09<00:25,  1.17it/s]